In [ ]:

import pandas as pd
from pathlib import Path

# 1. Dossiers des données
dossier = Path("../data/totalité_juillet_septembre_2025")

# 2. Charger les 16 pluviomètres déjà nettoyés et fusionnés
df = pd.read_csv(
    dossier / "pluviometres_16_fusionnes.csv",
    parse_dates=["date"]
)

# 3. Ajouter les stations d'oxygène dissous
for station in ["Q01", "Q02", "Q03", "Q04"]:

    fichier = dossier / "totalité_juillet_septembre_2025" / f"{station}.csv"

    if not fichier.exists():
        print(f"Attention : {station}.csv introuvable")
        continue

    oxygene = pd.read_csv(
        fichier,
        sep="\t",
        encoding="cp1252",
        decimal=","
    )

    # Garder la date et la mesure d'oxygène
    oxygene = oxygene.iloc[:, :2].copy()
    oxygene.columns = ["date", station]

    # Convertir les dates et les mesures
    oxygene["date"] = pd.to_datetime(
        oxygene["date"].astype(str),
        format="%d.%m.%Y %H:%M:%S",
        errors="coerce"
    )

    oxygene[station] = pd.to_numeric(
        oxygene[station],
        errors="coerce"
    )

    # Supprimer les dates invalides
    oxygene = oxygene.dropna(subset=["date"])

    # Éviter de multiplier les lignes en cas de doublons
    if oxygene["date"].duplicated().any():
        raise ValueError(f"Dates en double dans {station}.csv")

    # Fusionner selon la date
    df = df.merge(oxygene, on="date", how="outer")

# 4. Garder uniquement juillet, août et septembre 2025
df = df[
    (df["date"] >= "2025-07-01") &
    (df["date"] < "2025-10-01")
]

# 5. Trier chronologiquement
df = df.sort_values("date").reset_index(drop=True)

# 6. Vérifier le résultat
print("Nombre de lignes :", len(df))
print("Nombre de colonnes :", len(df.columns))
print("\nValeurs manquantes :")
print(df.isna().sum())

display(df.head())


Problemes apparus: 
1. L'encodage : Python ne reconnaît pas certains caractères accentués.
2. Le séparateur : Python coupe le nom date/time (SER UTC+2,00) au niveau de la virgule, alors que les colonnes des fichers d'oxygène sont séparées par des tabulations.  

Correction:  
  
   df_temp = pd.read_csv(. 
    fichier,  
    sep="\t",  
    encoding="cp1252"
) 

Á savori aussi: dans les fichers des pluviometres il y a plusieurs mesures. 
 
- La première mesure de pluie est une donnée brute, non corrigée.
- Une valeur supérieure à 10 mm en 5 minutes correspond à un étalonnage du pluviomètre, et non à une pluie réelle.
- Les colonnes pré-validées ou validées doivent être privilégiées lorsqu'elles sont disponibles et que leur signification est confirmée.  

Dans ce code: choix d'utiliser la mesure pré-validée lorsqu'elle existe ; sinon utiliser la mesure brute, mais remplacer les valeurs supérieures à 10 mm par une valeur manquante (NaN), jamais par zéro.

#### Verification données

In [ ]:
# Compter les valeurs manquantes par station
valeurs_manquantes = df.isna().sum()

print("Valeurs manquantes par station :")
print(valeurs_manquantes)

### Vérifier les 16 extraits

In [ ]:

import pandas as pd

# Les 16 extraits sélectionnés
extraits = [
    ("Avec baisse", "2025-07-06 12:00"),
    ("Avec baisse", "2025-07-20 17:00"),
    ("Avec baisse", "2025-07-24 19:00"),
    ("Avec baisse", "2025-08-02 14:00"),
    ("Avec baisse", "2025-08-31 17:00"),
    ("Avec baisse", "2025-09-03 22:00"),
    ("Avec baisse", "2025-09-15 07:00"),
    ("Avec baisse", "2025-09-19 21:00"),

    ("Sans baisse", "2025-07-10 12:00"),
    ("Sans baisse", "2025-07-17 06:00"),
    ("Sans baisse", "2025-07-24 12:00"),
    ("Sans baisse", "2025-08-10 12:00"),
    ("Sans baisse", "2025-08-20 12:00"),
    ("Sans baisse", "2025-08-31 06:00"),
    ("Sans baisse", "2025-09-22 12:00"),
    ("Sans baisse", "2025-09-25 12:00")
]

stations_oxygene = [
    c for c in ["Q01", "Q02", "Q03", "Q04"]
    if c in df.columns
]

stations_pluie = [
    c for c in df.columns
    if c.startswith("pluv")
]

resultats = []

for categorie, debut in extraits:

    debut = pd.Timestamp(debut)
    fin = debut + pd.Timedelta(hours=2)

    extrait = df[
        (df["date"] >= debut) &
        (df["date"] < fin)
    ]

    # Stations avec au moins une mesure sous 3 mg/L
    stations_basses = [
        station for station in stations_oxygene
        if (extrait[station] < 3).any()
    ]

    # Pluie cumulée par pluviomètre
    pluie_par_station = extrait[stations_pluie].sum(
        skipna=True,
        min_count=1
    )

    resultats.append({
        "Catégorie prévue": categorie,
        "Début": debut,
        "Mesures": len(extrait),
        "Stations sous 3 mg/L": ", ".join(stations_basses),
        "Pluviomètres avec pluie": int(
            (pluie_par_station > 0).sum()
        ),
        "Pluie maximale par station (mm)": pluie_par_station.max()
    })

verification = pd.DataFrame(resultats)

display(verification)


In [ ]:

# Vérifier les baisses d'oxygène d'au moins 15 minutes

resultats_oxygene = []

for categorie, debut in extraits:

    debut = pd.Timestamp(debut)
    fin = debut + pd.Timedelta(hours=2)

    extrait = df[
        (df["date"] >= debut) &
        (df["date"] < fin)
    ].sort_values("date")

    stations_critiques = []

    for station in stations_oxygene:

        # True lorsque l'oxygène est sous 3 mg/L
        sous_seuil = extrait[station] < 3

        # Identifier les séries de mesures consécutives
        groupes = sous_seuil.ne(sous_seuil.shift()).cumsum()

        # Durée de chaque série sous le seuil
        longueurs = sous_seuil.groupby(groupes).sum()

        # Au moins 3 mesures de 5 minutes = 15 minutes
        if (longueurs >= 3).any():
            stations_critiques.append(station)

    resultats_oxygene.append({
        "Début": debut,
        "Catégorie prévue": categorie,
        "Épisode ≥ 15 min": (
            "OUI" if stations_critiques else "NON"
        ),
        "Stations concernées": (
            ", ".join(stations_critiques)
            if stations_critiques else "Aucune"
        )
    })

verification_oxygene = pd.DataFrame(resultats_oxygene)

display(verification_oxygene)


In [ ]:

import pandas as pd
from pathlib import Path

# Dossier contenant tous les fichiers
dossier = Path("../data/totalité_juillet_septembre_2025")

# Charger les 16 pluviomètres déjà nettoyés
df = pd.read_csv(
    dossier / "pluviometres_16_fusionnes.csv",
    parse_dates=["date"]
)

# Ajouter les quatre stations d'oxygène
for station in ["Q01", "Q02", "Q03", "Q04"]:

    oxygene = pd.read_csv(
        dossier / f"{station}.csv",
        sep="\t",
        encoding="cp1252",
        decimal=","
    )

    oxygene = oxygene.iloc[:, :2].copy()
    oxygene.columns = ["date", station]

    oxygene["date"] = pd.to_datetime(
        oxygene["date"].astype(str),
        format="%d.%m.%Y %H:%M:%S",
        errors="coerce"
    )

    oxygene[station] = pd.to_numeric(
        oxygene[station],
        errors="coerce"
    )

    df = df.merge(oxygene, on="date", how="left")

# Vérification
print("Dimensions :", df.shape)

for station in ["Q01", "Q02", "Q03", "Q04"]:
    print(
        station,
        "| Minimum :", df[station].min(),
        "| Mesures sous 3 mg/L :", (df[station] < 3).sum()
    )

display(df.head())
